# Qwen Past-200 H100 Pipeline

Run top to bottom on Colab Pro with an H100 runtime. This notebook produces the artifacts for the past-200 push:

| Track | Artifact path | Gate |
|---|---|---|
| 1.5B single-session | `/content/drive/MyDrive/dismantle/qwen15b_artifacts` | projected `single_session_tps >= 200` |
| 3B aggregate | `/content/drive/MyDrive/dismantle/qwen3b_artifacts` | projected `aggregate_tps >= 200` |

The notebook trains Eagle5 candidates, runs tau/frontier search, emits AWQ and Q2/IQ2 calibration artifacts, and writes `past200_summary.md` with exact runtime hints.


In [ ]:
# Cell 1 - H100 setup, repo, knobs
from google.colab import drive
drive.mount('/content/drive')

import glob, gc, json, math, os, shutil, subprocess, sys
from pathlib import Path


def run(cmd, **kwargs):
    cmd = [str(x) for x in cmd]
    print('$ ' + ' '.join(cmd), flush=True)
    return subprocess.run(cmd, check=True, **kwargs)

run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'accelerate>=1.0', 'datasets>=3.0', 'pyarrow>=17', 'tqdm>=4.66',
    'zstandard', 'bitsandbytes>=0.43', 'gguf', 'safetensors>=0.4',
    'hf_transfer>=0.1.9', 'transformers>=4.45',
])

os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '1')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

try:
    from google.colab import userdata
    _hf_token = userdata.get('HF_TOKEN')
except Exception:
    _hf_token = None
if _hf_token:
    os.environ['HF_TOKEN'] = _hf_token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = _hf_token
    print('HF_TOKEN loaded from Colab secrets.')
else:
    print('No HF_TOKEN secret found; public Hub downloads still work.')

import numpy as np
import pyarrow
import torch
import transformers
from transformers import AutoConfig

assert torch.cuda.is_available(), 'No CUDA device: choose an H100 GPU runtime.'
GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
STRICT_H100 = True
if STRICT_H100 and 'H100' not in GPU_NAME.upper():
    raise RuntimeError(f'This notebook is H100-tuned; current GPU is {GPU_NAME}. Set STRICT_H100=False to override.')
print(f'GPU: {GPU_NAME}  VRAM: {VRAM_GB:.1f} GB')
print(f'transformers={transformers.__version__} torch={torch.__version__} pyarrow={pyarrow.__version__}')

DRIVE_ROOT = '/content/drive/MyDrive/dismantle'
LOCAL_ROOT = '/content/dismantle_work'
REPO_DIR = '/content/dismantle'
REPO = 'https://github.com/joshuahickscorp/dismantle.git'
BRANCH = 'main'

RUN_STUDENT_SINGLE_SESSION = True
RUN_QWEN3B_AGGREGATE = True
BUILD_CORPUS_IF_NEEDED = True
FORCE_RETRAIN = False
FORCE_TAU = False
FORCE_FRONTIER = False
USE_TORCH_COMPILE = True

TARGET_TPS = 200.0
CB_SLOTS = 4
CB_EFFICIENCY = 0.74
SPEC_EFFICIENCY_STUDENT = 0.84
SPEC_EFFICIENCY_QWEN3B = 0.82

TARGETS = [
    {
        'name': 'qwen15b',
        'enabled': RUN_STUDENT_SINGLE_SESSION,
        'model_id': 'Qwen/Qwen2.5-1.5B-Instruct',
        'artifact_dir': f'{DRIVE_ROOT}/qwen15b_artifacts',
        'corpus_dir': f'{DRIVE_ROOT}/qwen15b_corpus',
        'local_corpus': f'{LOCAL_ROOT}/qwen15b_corpus',
        'local_frozen': '/content/qwen15b_frozen.npz',
        'corpus_target_seqs': 12000,
        'train_max_rows': 12000,
        'train_max_row_tokens': 384,
        'train_epochs': 12,
        'train_batch': 48,
        'eval_max_windows': 16000,
        'frontier_max_depth': 16,
        'frontier_top_n': 4,
        'base_tps': 85.0,
        'quant_multiplier': 1.00,
        'spec_efficiency': SPEC_EFFICIENCY_STUDENT,
        'profile_hint': 'qwen15b-q2-or-iq2',
    },
    {
        'name': 'qwen3b',
        'enabled': RUN_QWEN3B_AGGREGATE,
        'model_id': 'Qwen/Qwen2.5-3B-Instruct',
        'artifact_dir': f'{DRIVE_ROOT}/qwen3b_artifacts',
        'corpus_dir': f'{DRIVE_ROOT}/qwen3b_corpus',
        'local_corpus': f'{LOCAL_ROOT}/qwen3b_corpus',
        'local_frozen': '/content/qwen3b_frozen.npz',
        'corpus_target_seqs': 10000,
        'train_max_rows': 10000,
        'train_max_row_tokens': 320,
        'train_epochs': 10,
        'train_batch': 24,
        'eval_max_windows': 12000,
        'frontier_max_depth': 12,
        'frontier_top_n': 3,
        'base_tps': 26.6,
        'quant_multiplier': 1.25,
        'spec_efficiency': SPEC_EFFICIENCY_QWEN3B,
        'profile_hint': 'qwen3b-q4k-w4a8',
    },
]

TRAIN_SPECS = [
    {'family': 'b1', 'num_blocks': 1, 'lr': 3e-4, 'residual_delta': 0.00, 'calib_weight': 0.10},
    {'family': 'b1', 'num_blocks': 1, 'lr': 6e-4, 'residual_delta': 0.00, 'calib_weight': 0.12},
    {'family': 'b1', 'num_blocks': 1, 'lr': 1e-3, 'residual_delta': 0.00, 'calib_weight': 0.12},
    {'family': 'b2', 'num_blocks': 2, 'lr': 3e-4, 'residual_delta': 0.01, 'calib_weight': 0.16},
    {'family': 'b2', 'num_blocks': 2, 'lr': 6e-4, 'residual_delta': 0.02, 'calib_weight': 0.18},
    {'family': 'b2', 'num_blocks': 2, 'lr': 1e-3, 'residual_delta': 0.02, 'calib_weight': 0.20},
]

for d in (DRIVE_ROOT, LOCAL_ROOT):
    os.makedirs(d, exist_ok=True)
for t in TARGETS:
    for key in ('artifact_dir', 'corpus_dir', 'local_corpus'):
        os.makedirs(t[key], exist_ok=True)

if not os.path.exists(REPO_DIR):
    run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO, REPO_DIR])
else:
    run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH, '--depth', '1'])
    run(['git', '-C', REPO_DIR, 'checkout', BRANCH])
    run(['git', '-C', REPO_DIR, 'reset', '--hard', f'origin/{BRANCH}'])
os.chdir(REPO_DIR)
print('repo:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())

REQUIRED_FILES = [
    'colab/mega_calibrate.py',
    'colab/eagle5_train_pytorch.py',
    'colab/eagle5_tau_eval_pytorch.py',
    'colab/eagle5_frontier_policy.py',
    'colab/awq_w4a8_validate.py',
    'colab/build_qwen3b_frozen_hf.py',
    'colab/q2k_importance_calibrate.py',
    'tools/training/awq_calibrate.py',
]
for path in REQUIRED_FILES:
    assert os.path.exists(path), f'missing required file: {path}'
    print('ok', path)

print(json.dumps({
    'target_tps': TARGET_TPS,
    'cb_slots': CB_SLOTS,
    'cb_efficiency': CB_EFFICIENCY,
    'train_specs': TRAIN_SPECS,
    'targets': [{k: v for k, v in t.items() if k != 'enabled' or v} for t in TARGETS],
}, indent=2))


In [ ]:
# Cell 2 - Helpers

def target_config(target):
    cfg = AutoConfig.from_pretrained(target['model_id'], trust_remote_code=False)
    n_layers = int(getattr(cfg, 'num_hidden_layers', 0) or getattr(cfg, 'n_layer', 0))
    hidden = int(getattr(cfg, 'hidden_size', 0) or getattr(cfg, 'n_embd', 0))
    vocab = int(getattr(cfg, 'vocab_size', 0))
    capture_layer = max(0, n_layers - 4)
    return {'n_layers': n_layers, 'hidden': hidden, 'vocab': vocab, 'capture_layer': capture_layer}


def stats_sequences(path):
    if not os.path.exists(path):
        return 0
    try:
        with np.load(path) as z:
            return int(np.asarray(z['sequences_written']).item()) if 'sequences_written' in z.files else 0
    except Exception as e:
        print(f'WARN: could not read stats file {path}: {e}')
        return 0


def corpus_state(target):
    stats_file = f"{target['corpus_dir']}/per_site_activation_stats.npz"
    shards = len(glob.glob(f"{target['corpus_dir']}/shard_*.parquet"))
    seqs = stats_sequences(stats_file)
    required_shards = math.ceil(target['corpus_target_seqs'] / 16)
    return {
        'shards': shards,
        'seqs': seqs,
        'required_shards': required_shards,
        'complete': shards >= required_shards and seqs >= target['corpus_target_seqs'],
    }


def run_calibration(target, info):
    state = corpus_state(target)
    print(f"[{target['name']}] corpus state: {state}")
    if state['complete']:
        return
    if not BUILD_CORPUS_IF_NEEDED:
        raise RuntimeError(f"{target['name']} corpus incomplete and BUILD_CORPUS_IF_NEEDED=False")

    if target['name'] == 'qwen15b':
        batch, lm_head_chunk = 16, 512
    else:
        batch, lm_head_chunk = 10, 384
    run([
        sys.executable, 'colab/mega_calibrate.py',
        '--model', target['model_id'],
        '--max-sequences', str(target['corpus_target_seqs']),
        '--max-tokens', '2048',
        '--batch-size', str(batch),
        '--capture-layer', str(info['capture_layer']),
        '--topk-logits', '100',
        '--lm-head-chunk-tokens', str(lm_head_chunk),
        '--shard-size', '16',
        '--sync-dir', target['corpus_dir'],
        '--sync-every', '4',
        '--delete-local-after-sync',
        '--out', target['local_corpus'],
    ])
    print(f"[{target['name']}] corpus state after calibration: {corpus_state(target)}")


def build_quant_artifacts(target):
    stats_file = f"{target['corpus_dir']}/per_site_activation_stats.npz"
    awq_out = f"{target['artifact_dir']}/{target['name']}_awq_smoothing.json"
    q2_out = f"{target['artifact_dir']}/{target['name']}_q2k_importance.npz"
    assert os.path.exists(stats_file), f'missing activation stats: {stats_file}'
    run([
        sys.executable, 'tools/training/awq_calibrate.py',
        '--stats', stats_file,
        '--out', awq_out,
        '--alpha', '0.5',
        '--model', target['model_id'],
    ])
    run([
        sys.executable, 'colab/q2k_importance_calibrate.py',
        '--stats', stats_file,
        '--out', q2_out,
        '--model', target['model_id'],
        '--mean-weight', '0.70',
        '--max-weight', '0.30',
    ])
    return {'stats': stats_file, 'awq': awq_out, 'q2_importance': q2_out}


def build_frozen(target):
    frozen_out = f"{target['artifact_dir']}/{target['name']}_frozen.npz"
    run([
        sys.executable, 'colab/build_qwen3b_frozen_hf.py',
        '--model', target['model_id'],
        '--out', frozen_out,
    ])
    assert os.path.exists(frozen_out), f'missing frozen baseline: {frozen_out}'
    return frozen_out


def lr_tag(lr):
    return f'{lr:.0e}'.replace('+', '').replace('-0', '-')


def spec_tag(spec):
    rd = int(round(spec['residual_delta'] * 1000))
    cw = int(round(spec['calib_weight'] * 100))
    return f"{spec['family']}_blocks{spec['num_blocks']}_lr{lr_tag(spec['lr'])}_rd{rd:03d}_cw{cw:02d}"


def stage_frozen(target, frozen_out):
    if not os.path.exists(target['local_frozen']) or os.path.getsize(target['local_frozen']) != os.path.getsize(frozen_out):
        print(f"[{target['name']}] stage frozen -> {target['local_frozen']}")
        shutil.copyfile(frozen_out, target['local_frozen'])
    return target['local_frozen']


In [ ]:
# Cell 3 - Build corpora, AWQ, Q2/IQ2 importance, frozen baselines
TARGET_INFO = {}
TARGET_ARTIFACTS = {}
for target in TARGETS:
    if not target['enabled']:
        continue
    info = target_config(target)
    TARGET_INFO[target['name']] = info
    print(f"\n=== {target['name']} {target['model_id']} cfg={info}")
    run_calibration(target, info)
    quant = build_quant_artifacts(target)
    frozen = build_frozen(target)
    TARGET_ARTIFACTS[target['name']] = {**quant, 'frozen': frozen}
    print(json.dumps(TARGET_ARTIFACTS[target['name']], indent=2))


In [ ]:
# Cell 4 - Train Eagle5 grids and run tau/frontier ranking
ALL_RESULTS = {}
for target in TARGETS:
    if not target['enabled']:
        continue
    name = target['name']
    info = TARGET_INFO[name]
    artifacts = TARGET_ARTIFACTS[name]
    local_frozen = stage_frozen(target, artifacts['frozen'])
    gc.collect()
    torch.cuda.empty_cache()

    trained = []
    for spec in TRAIN_SPECS:
        tag = spec_tag(spec)
        ckpt_dir = f"{target['artifact_dir']}/checkpoints/eagle5_{name}_{tag}"
        final_head = f'{ckpt_dir}/head_final.safetensors'
        os.makedirs(ckpt_dir, exist_ok=True)
        if os.path.exists(final_head) and not FORCE_RETRAIN:
            print(f"[{name}] skip trained {tag}: {final_head}")
        else:
            print(f"\n=== [{name}] train {tag} spec={spec}")
            gc.collect()
            torch.cuda.empty_cache()
            cmd = [
                sys.executable, 'colab/eagle5_train_pytorch.py',
                '--corpus-dir', target['corpus_dir'],
                '--frozen', local_frozen,
                '--ckpt-dir', ckpt_dir,
                '--epochs', str(target['train_epochs']),
                '--batch-size', str(target['train_batch']),
                '--seq-len', '16',
                '--lr', str(spec['lr']),
                '--num-blocks', str(spec['num_blocks']),
                '--head-heads', '16',
                '--capture-layer', str(info['capture_layer']),
                '--max-rows', str(target['train_max_rows']),
                '--max-row-tokens', str(target['train_max_row_tokens']),
                '--sparsity-head', 'off',
                '--seed', '0',
                '--calib-loss-weight', str(spec['calib_weight']),
                '--residual-delta-loss-weight', str(spec['residual_delta']),
                '--save-safetensors',
            ]
            if USE_TORCH_COMPILE:
                cmd.append('--compile')
            run(cmd)
        assert os.path.exists(final_head), f'training did not write {final_head}'
        trained.append((tag, spec, final_head))

    tau_results = {}
    for tag, spec, ckpt in trained:
        out_path = f"{target['artifact_dir']}/eagle5_tau_{name}_{tag}.json"
        if os.path.exists(out_path) and not FORCE_TAU:
            print(f"[{name}] skip tau {tag}: {out_path}")
        else:
            print(f"\n=== [{name}] tau eval {tag}")
            gc.collect()
            torch.cuda.empty_cache()
            run([
                sys.executable, 'colab/eagle5_tau_eval_pytorch.py',
                '--ckpt', ckpt,
                '--frozen', local_frozen,
                '--corpus', target['corpus_dir'],
                '--out', out_path,
                '--depth', '4',
                '--max-windows', str(target['eval_max_windows']),
                '--max-row-tokens', str(target['train_max_row_tokens']),
                '--base-tps', str(target['base_tps']),
                '--w4a8-multiplier', str(target['quant_multiplier']),
                '--spec-efficiency', str(target['spec_efficiency']),
            ])
        with open(out_path) as f:
            tau_results[tag] = json.load(f)
        tau_results[tag]['train_spec'] = spec
        tau_results[tag]['ckpt'] = ckpt

    ranked_tau = sorted(
        tau_results.items(),
        key=lambda kv: (kv[1]['projection']['projected_dec_tps'], kv[1]['tau'], kv[1]['depth1_accept_rate']),
        reverse=True,
    )
    tau_winner_path = f"{target['artifact_dir']}/eagle5_tau_winner_{name}.json"
    with open(tau_winner_path, 'w') as f:
        json.dump({'winner': ranked_tau[0][0], 'result': ranked_tau[0][1], 'all': tau_results}, f, indent=2, sort_keys=True)

    print(f"\n[{name}] Tau race results:")
    for tag, r in ranked_tau:
        print(f"  {tag:34s} tau={r['tau']:.3f} depth1={r['depth1_accept_rate']:.2%} projected={r['projection']['projected_dec_tps']:.1f}")

    frontier_results = {}
    for tag, r in ranked_tau[:target['frontier_top_n']]:
        out_path = f"{target['artifact_dir']}/eagle5_frontier_{name}_{tag}.json"
        if os.path.exists(out_path) and not FORCE_FRONTIER:
            print(f"[{name}] skip frontier {tag}: {out_path}")
        else:
            print(f"\n=== [{name}] frontier search {tag}")
            run([
                sys.executable, 'colab/eagle5_frontier_policy.py',
                '--ckpt', r['ckpt'],
                '--frozen', local_frozen,
                '--corpus', target['corpus_dir'],
                '--out', out_path,
                '--max-depth', str(target['frontier_max_depth']),
                '--depths', '4,6,8,12,16',
                '--lattice-widths', '2,3,4,6',
                '--max-windows', str(target['eval_max_windows']),
                '--max-row-tokens', str(target['train_max_row_tokens']),
                '--eval-batch-size', '192',
                '--base-tps', str(target['base_tps']),
                '--w4a8-multiplier', str(target['quant_multiplier']),
                '--spec-efficiency', str(target['spec_efficiency']),
            ])
        with open(out_path) as f:
            frontier_results[tag] = json.load(f)

    frontier_ranked = sorted(
        frontier_results.items(),
        key=lambda kv: kv[1]['policies']['best_deployable']['projected_dec_tps'],
        reverse=True,
    )
    frontier_winner_path = f"{target['artifact_dir']}/eagle5_frontier_winner_{name}.json"
    with open(frontier_winner_path, 'w') as f:
        json.dump({'winner': frontier_ranked[0][0], 'result': frontier_ranked[0][1], 'all': frontier_results}, f, indent=2, sort_keys=True)

    print(f"\n[{name}] Frontier results:")
    for tag, payload in frontier_ranked:
        best = payload['policies']['best_deployable']
        overall = payload['policies']['best_overall']
        print(f"  {tag:34s} deployable={best['kind']:34s} projected={best['projected_dec_tps']:.1f} overall={overall['projected_dec_tps']:.1f}")

    ALL_RESULTS[name] = {
        'target': target,
        'info': info,
        'artifacts': artifacts,
        'tau_winner_path': tau_winner_path,
        'frontier_winner_path': frontier_winner_path,
        'ranked_tau': ranked_tau,
        'frontier_ranked': frontier_ranked,
    }


In [ ]:
# Cell 5 - Past-200 summary and local handoff commands
summary_path = f'{DRIVE_ROOT}/past200_summary.md'
json_summary_path = f'{DRIVE_ROOT}/past200_summary.json'
summary = {
    'schema': 'dismantle-past200-h100-v1',
    'gpu': GPU_NAME,
    'target_tps': TARGET_TPS,
    'cb_slots': CB_SLOTS,
    'cb_efficiency': CB_EFFICIENCY,
    'targets': {},
}

lines = [
    '# Past-200 H100 Summary',
    '',
    f'GPU: `{GPU_NAME}`',
    f'Target TPS: `{TARGET_TPS:.1f}`',
    '',
    '## Results',
    '',
    '| Target | Track | Winner | Policy | Projected TPS | Gate |',
    '|---|---|---|---|---:|---|',
]

for name, result in ALL_RESULTS.items():
    target = result['target']
    frontier_info = json.load(open(result['frontier_winner_path']))
    tau_info = json.load(open(result['tau_winner_path']))
    winner_tag = frontier_info['winner']
    policy = frontier_info['result']['policies']['best_deployable']
    hints = frontier_info['result']['runtime_hints']
    per_session = float(policy['projected_dec_tps'])
    aggregate = per_session * CB_SLOTS * CB_EFFICIENCY

    track = 'single-session' if name == 'qwen15b' else 'aggregate'
    projected = per_session if name == 'qwen15b' else aggregate
    gate = 'PASS' if projected >= TARGET_TPS else 'MISS'
    lines.append(f'| `{name}` | {track} | `{winner_tag}` | `{policy["kind"]}` | {projected:.1f} | `{gate}` |')

    summary['targets'][name] = {
        'track': track,
        'model_id': target['model_id'],
        'profile_hint': target['profile_hint'],
        'winner': winner_tag,
        'tau_winner': tau_info['winner'],
        'best_policy': policy,
        'runtime_hints': hints,
        'per_session_tps': per_session,
        'aggregate_tps': aggregate,
        'projected_gate_tps': projected,
        'passed': projected >= TARGET_TPS,
        'artifacts': result['artifacts'],
        'frontier_winner_path': result['frontier_winner_path'],
        'tau_winner_path': result['tau_winner_path'],
    }

lines.extend(['', '## Runtime Hints', ''])
for name, payload in summary['targets'].items():
    lines.extend([
        f'### {name}',
        '',
        '```json',
        json.dumps(payload['runtime_hints'], indent=2, sort_keys=True),
        '```',
        '',
    ])

lines.extend(['## Artifacts', ''])
for name, payload in summary['targets'].items():
    lines.append(f'### {name}')
    for key, path in payload['artifacts'].items():
        size = os.path.getsize(path) / 1e6 if os.path.exists(path) else 0.0
        lines.append(f'- `{key}`: `{path}` ({size:.1f} MB)')
    lines.append(f'- `tau`: `{payload["tau_winner_path"]}`')
    lines.append(f'- `frontier`: `{payload["frontier_winner_path"]}`')
    lines.append('')

student = summary['targets'].get('qwen15b')
qwen3b = summary['targets'].get('qwen3b')
lines.extend(['## Local Commands', ''])
if student:
    s_art = student['artifacts']
    lines.extend([
        '### Single-session student bench',
        '',
        '```bash',
        f'DISMANTLE_QWEN_AWQ=1 \\',
        f'DISMANTLE_QWEN_AWQ_SMOOTHING=profiles/qwen15b_awq_smoothing.json \\',
        f'DISMANTLE_Q2K_IMPORTANCE=artifacts/qwen15b_q2k_importance.npz \\',
        f'EAGLE5_HEAD=artifacts/qwen15b_artifacts/checkpoints/eagle5_qwen15b_{student["winner"]}/head_final.safetensors \\',
        'TRIALS=10 TOKENS=128 \\',
        '  bash tools/bench/eagle5_paired_bench.sh',
        '```',
        '',
    ])
if qwen3b:
    lines.extend([
        '### Aggregate continuous-batching serve probe',
        '',
        '```bash',
        f'DISMANTLE_CB_ALPHA=1 \\',
        f'DISMANTLE_MAX_BATCH_SIZE={CB_SLOTS} \\',
        f'DISMANTLE_SPEC_DECODE=eagle5 \\',
        f'EAGLE5_HEAD=artifacts/qwen3b_artifacts/checkpoints/eagle5_qwen3b_{qwen3b["winner"]}/head_final.safetensors \\',
        '  cargo run --release -p dismantle -- serve models/qwen2.5-3b-instruct-q4_k_m.gguf',
        '```',
        '',
    ])

with open(summary_path, 'w') as f:
    f.write('\n'.join(lines) + '\n')
with open(json_summary_path, 'w') as f:
    json.dump(summary, f, indent=2, sort_keys=True)

print('\n'.join(lines))
print(f'\nsummary: {summary_path}')
print(f'json: {json_summary_path}')
